# Stochastic PyPSA network and SMS++ conversion step by step

This notebook follows the workflow in `main_stochastic.py` and makes it explicit. The goal is to build a PyPSA network from the Excel file `test/configs/data/develop/stochastic_2n_2g_maxpower.xlsx`, make it stochastic, optimize it with PyPSA, and then convert/solve it with `pypsa2smspp`.

Unlike the original script, this notebook does not use the `NetworkDefinition` class: every network-building step is visible here.

## 1. Imports and paths

The notebook can be run either from the repository root or from the `test` folder. The `application_stochastic.ini` configuration is used only to retrieve paths and stochastic parameters, not to build the network automatically.

In [ ]:
from configparser import ConfigParser
from pathlib import Path
import ast
import os
import sys

import numpy as np
import pandas as pd
import pypsa

# Find the repository root starting from the notebook's current directory.
cwd = Path.cwd().resolve()
REPO_ROOT = next(
    p for p in [cwd, *cwd.parents]
    if (p / "pypsa2smspp").is_dir() and (p / "test").is_dir()
)
TEST_DIR = REPO_ROOT / "test"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(TEST_DIR) not in sys.path:
    sys.path.insert(0, str(TEST_DIR))

from pypsa2smspp.network_correction import add_slack_unit
from pypsa2smspp.transformation import Transformation

CONFIG_PATH = TEST_DIR / "configs" / "application_stochastic.ini"
config = ConfigParser()
config.read(CONFIG_PATH)

INPUT_DIR = (CONFIG_PATH.parent / config["INPUT"]["input_data_path"]).resolve()
EXCEL_PATH = INPUT_DIR / config["INPUT"]["input_name_components"]
DEMAND_PATH = INPUT_DIR / config["INPUT"]["input_name_demand"]

STOCHASTIC_PARAMETERS = ast.literal_eval(config["INPUT"]["stochastic_parameters"])
N_SNAPSHOTS = int(config["OPTIMIZATION"]["n_snapshots"])
SNAPSHOT_WEIGHT = float(config["OPTIMIZATION"]["weight"])
LOAD_SIGN = float(config["NETWORK_DEFINITION"]["load_sign"])

EXCEL_PATH

## 2. Reading the Excel file

The Excel file contains sheets named after PyPSA components (`Bus`, `Load`, `Line`, `Generator`, etc.) and sheets for time series, for example `generators_t_p_max_pu`. Here we read all sheets and remove fully empty rows/columns.

In [ ]:
def clean_sheet(df: pd.DataFrame) -> pd.DataFrame:
    """Remove empty rows/columns produced by Excel editing."""
    return df.dropna(axis=0, how="all").dropna(axis=1, how="all").copy()


sheets = {name: clean_sheet(df) for name, df in pd.read_excel(EXCEL_PATH, sheet_name=None).items()}
[(name, df.shape) for name, df in sheets.items()]

## 3. Snapshots

If the `snapshots` sheet exists, we use it. Timestamps come from Excel: if they are serial numbers, we convert them to dates; if they are already dates, we keep them as they are. Snapshot weightings are read from the same sheet.

In [ ]:
def excel_serial_to_datetime(values: pd.Series) -> pd.Index:
    if pd.api.types.is_numeric_dtype(values):
        return pd.to_datetime(values, unit="D", origin="1899-12-30")
    return pd.Index(values)


n = pypsa.Network()

if "snapshots" in sheets and not sheets["snapshots"].empty:
    snapshot_df = sheets["snapshots"]
    snapshots = excel_serial_to_datetime(snapshot_df["snapshot"])
    n.set_snapshots(snapshots)

    for col in ["objective", "generators", "stores"]:
        if col in snapshot_df.columns:
            n.snapshot_weightings.loc[n.snapshots, col] = snapshot_df[col].to_numpy()
else:
    n.set_snapshots(range(N_SNAPSHOTS))
    n.snapshot_weightings.loc[:, "objective"] = SNAPSHOT_WEIGHT
    n.snapshot_weightings.loc[:, "generators"] = SNAPSHOT_WEIGHT

n.snapshots[:5]

## 4. Static components

We now manually add the static components. This is intentionally explicit: every non-empty row in the Excel component sheets is represented by its own `n.add(...)` call. In this Excel file, the `Link`, `StorageUnit`, and `Store` sheets do not contain active components.

In [ ]:
# Carriers
n.add("Carrier", "AC", color="#16ba24")
n.add("Carrier", "solar", color="#f5b111")
n.add("Carrier", "slack", color="#adab00")
n.add("Carrier", "electrical grid selling", color="#f7f300")
n.add("Carrier", "gas", color="#6b4c04")
n.add("Carrier", "gas CHP", color="#210af5")
n.add("Carrier", "hydro", color="#0cdaf5")
n.add("Carrier", "battery", color="#16942d")
n.add("Carrier", "thermal storage", color="#c904a9")
n.add("Carrier", "resistive heater", color="#66039c")
n.add("Carrier", "wind", color="#66039c")
n.add("Carrier", "battery_link", color="#66039c")
n.add("Carrier", "H2", color="#16942d")
n.add("Carrier", "H2 electrolysis", color="#16942d")
n.add("Carrier", "H2 fuel cell", color="#16942d")
n.add("Carrier", "diesel", color="#6b4c04")

# Buses
n.add(
    "Bus",
    "IT0 0",
    v_nom=380,
    x=11.861807419676801,
    y=43.502700216278299,
    carrier="AC",
    v_mag_pu_set=1,
    v_mag_pu_min=0,
    v_mag_pu_max=np.inf,
    control="Slack",
    substation_off=1,
    country="IT",
    substation_lv=1,
)
n.add(
    "Bus",
    "IT0 1",
    v_nom=380,
    x=15,
    y=50,
    carrier="AC",
    v_mag_pu_set=1,
    v_mag_pu_min=0,
    v_mag_pu_max=np.inf,
    control="Slack",
    substation_off=1,
    country="IT",
    substation_lv=1,
)

# Loads
n.add("Load", "IT0 0", carrier="AC", bus="IT0 0")
n.add("Load", "IT0 1", carrier="AC", bus="IT0 1")

# Lines
n.add(
    "Line",
    "Line 0-1",
    carrier="AC",
    bus0="IT0 0",
    bus1="IT0 1",
    s_nom_extendable=True,
    x=5,
    r=0.01,
    s_nom=10,
)

# Generators
n.add(
    "Generator",
    "IT0 0 0 solar",
    bus="IT0 0",
    control="Slack",
    p_nom=20326.975357459101,
    p_nom_mod=0,
    p_nom_extendable=True,
    p_nom_min=20326.975357459101,
    p_nom_max=725545.55299178895,
    p_min_pu=0,
    p_max_pu=1,
    e_sum_max=np.inf,
    q_set=0,
    sign=1,
    carrier="solar",
    marginal_cost=0.01,
    marginal_cost_quadratic=0,
    active=True,
    build_year=0,
    lifetime=40,
    capital_cost=1189.6300057910801,
    efficiency=1,
    committable=False,
    start_up_cost=0,
    shut_down_cost=0,
    stand_by_cost=0,
    min_up_time=0,
    min_down_time=0,
    up_time_before=1,
    down_time_before=0,
    ramp_limit_start_up=1,
    ramp_limit_shut_down=1,
    weight=1,
)
n.add(
    "Generator",
    "IT0 0 0 onwind",
    bus="IT0 0",
    control="PQ",
    p_nom=9582.6490542581105,
    p_nom_mod=0,
    p_nom_extendable=True,
    p_nom_min=9582.6490542581105,
    p_nom_max=455771.75449044502,
    p_min_pu=0,
    p_max_pu=1,
    e_sum_max=np.inf,
    q_set=0,
    sign=1,
    carrier="onwind",
    marginal_cost=0.015,
    marginal_cost_quadratic=0,
    active=True,
    build_year=0,
    lifetime=30,
    capital_cost=108204.66880406901,
    efficiency=1,
    committable=False,
    start_up_cost=0,
    shut_down_cost=0,
    stand_by_cost=0,
    min_up_time=0,
    min_down_time=0,
    up_time_before=1,
    down_time_before=0,
    ramp_limit_start_up=1,
    ramp_limit_shut_down=1,
    weight=1,
)

n

## 5. Time series

The `generators_t_p_max_pu` sheet contains the maximum output profile for the solar generator. The Excel file does not include `loads_t_p_set`, so we explicitly reproduce the fallback used by the original script: the load profile is read from `demand_data.csv` and assigned to each load.

In [ ]:
# Solar availability profile, read from the dedicated Excel time-series sheet.
solar_p_max_pu = sheets["generators_t_p_max_pu"].copy()
solar_p_max_pu.index = n.snapshots
n.generators_t.p_max_pu["IT0 0 0 solar"] = solar_p_max_pu["IT0 0 0 solar"].to_numpy()

# The Excel file has no loads_t_p_set sheet. The original NetworkDefinition
# therefore falls back to demand_data.csv; here we spell that out directly.
demand_day = pd.read_csv(DEMAND_PATH)
n_days = int(np.ceil(len(n.snapshots) / 24))
demand_mean = np.tile(demand_day["demand"].to_numpy(), n_days)[: len(n.snapshots)]
demand_std = np.tile(demand_day["standard_deviation"].to_numpy(), n_days)[: len(n.snapshots)]

rng = np.random.default_rng(123)
load_it0_0 = LOAD_SIGN * rng.normal(demand_mean, demand_std) * 100
load_it0_1 = LOAD_SIGN * rng.normal(demand_mean, demand_std) * 100

n.loads_t.p_set["IT0 0"] = load_it0_0
n.loads_t.p_set["IT0 1"] = load_it0_1

n.loads_t.p_set.head(), n.generators_t.p_max_pu.head()

## 6. Slack unit

As in `main_stochastic.py`, we add a slack unit to buses that host loads. This helps keep the model feasible by allowing expensive emergency generation.

In [ ]:
n = add_slack_unit(n)
n.generators.tail()

## 7. Stochastic scenarios

PyPSA handles scenarios through `set_scenarios`. We keep three equally likely scenarios: `low`, `med`, and `high`. For the parameters configured in `application_stochastic.ini`, we modify the time series scenario by scenario.

In [ ]:
SCENARIOS = ["low", "med", "high"]
PROBABILITIES = {scenario: 1 / len(SCENARIOS) for scenario in SCENARIOS}

base_load = n.loads_t.p_set.copy()
base_p_max_pu = n.generators_t.p_max_pu.copy()

n.set_scenarios(PROBABILITIES)

for parameter in STOCHASTIC_PARAMETERS:
    if parameter == "demand":
        load_by_scenario = {
            "low": base_load,
            "med": base_load * 2,
            "high": base_load * 4,
        }
        for scenario in SCENARIOS:
            n.loads_t.p_set[scenario] = load_by_scenario[scenario]

    elif parameter == "renewables":
        pmax_by_scenario = {
            "low": base_p_max_pu / 2,
            "med": base_p_max_pu,
            "high": base_p_max_pu * 2 / 3,
        }
        for scenario in SCENARIOS:
            n.generators_t.p_max_pu[scenario] = pmax_by_scenario[scenario]

n.scenarios

## 8. Ottimizzazione PyPSA

Questa e' la soluzione di riferimento. Lo script originale usa Gurobi; se si vuole usare HiGHS, basta cambiare `PYPSA_SOLVER`.

In [ ]:
PYPSA_SOLVER = "gurobi"

n_pypsa = n.copy()
n_pypsa.optimize(solver_name=PYPSA_SOLVER)

obj_pypsa = float(n_pypsa.objective + n_pypsa.objective_constant)
obj_pypsa

## 9. Conversione e ottimizzazione SMS++

We now pass the stochastic network to `Transformation`. We use the same options as the script: the `TSSBlock/TSSBSCfg_grb.txt` configuration, thermal units disabled, and a `tssb` stochastic transformation.

In [ ]:
NAME = "stochastic_2intermittent_2datamapping_notebook"
FOLDER = "develop/tssb_notebook"
WORKDIR = TEST_DIR / "output" / FOLDER
WORKDIR.mkdir(parents=True, exist_ok=True)

transformation = Transformation(
    name=NAME,
    configfile="TSSBlock/TSSBSCfg_grb.txt",
    enable_thermal_units=False,
    workdir=WORKDIR,
    stochastic_parameters={
        "stochastic_type": "tssb",
        "parameters": STOCHASTIC_PARAMETERS,
    },
)

n_smspp = transformation.run(n.copy())
obj_smspp = float(n_smspp.objective)
obj_smspp

## 10. Confronto e salvataggio

We compare the PyPSA and SMS++ objectives and save the resulting networks as NetCDF files, as in the original script.

In [ ]:
error_percent = (obj_smspp - obj_pypsa) / obj_pypsa * 100
print(f"Objective PyPSA: {obj_pypsa:.6f}")
print(f"Objective SMS++: {obj_smspp:.6f}")
print(f"PyPSA-SMS++ error: {error_percent:.6f}%")

n_pypsa.export_to_netcdf(WORKDIR / f"pypsa_{NAME}.nc")
n_smspp.export_to_netcdf(WORKDIR / f"smspp_{NAME}.nc")

if hasattr(n_pypsa, "model"):
    n_pypsa.model.to_file(fn=WORKDIR / f"pypsa_{NAME}.lp")

WORKDIR